# CIFAR-100 — Wide ResNet-16-4, thử 20 epoch

Notebook này tự chứa toàn bộ mã train, không cần file Python bên ngoài. Model được khởi tạo ngẫu nhiên, không dùng trọng số pretrained. Dữ liệu đọc từ thư mục `cifar-100` cạnh project.

Train: flip ngang + crop 32×32 từ ảnh pad 4 pixel + color jitter nhẹ. Sau đó chọn Cutout (25%), Random Erasing (25%) hoặc giữ nguyên. Mỗi batch chọn MixUp (30%), CutMix (30%) hoặc giữ nguyên. Các phép mạnh được chọn luân phiên để tránh che/trộn quá nhiều trên ảnh nhỏ. Validation và test không dùng augmentation; 20 epoch chỉ dùng validation để đánh giá và chọn checkpoint.


## Cấu hình và đọc dữ liệu


In [1]:
from __future__ import annotations

import math
import pickle
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import tensorflow as tf

PROJECT_DIR = next(
    (path.resolve() for path in (Path.cwd(), Path.cwd().parent, Path.cwd() / "fruit-detection")
     if (path / "cifar-100" / "train").is_file()),
    Path.cwd().resolve(),
)
IMAGE_SIZE = 32
NUM_CLASSES = 100

@dataclass(frozen=True)
class TrainConfig:
    epochs: int = 20
    batch_size: int = 64
    seed: int = 42
    validation_size: int = 5_000
    mix_probability: float = 0.30
    cutmix_probability: float = 0.30
    cutout_probability: float = 0.25
    erasing_probability: float = 0.25
    mix_alpha: float = 0.5
    cutmix_alpha: float = 1.0

def load_cifar100(data_dir: Path = PROJECT_DIR / "cifar-100"):
    def load_batch(path: Path):
        with path.open("rb") as stream:
            data = pickle.load(stream, encoding="bytes")
        images = np.asarray(data[b"data"], dtype=np.uint8).reshape(-1, 3, 32, 32)
        images = images.transpose(0, 2, 3, 1).copy()
        labels = np.asarray(data[b"fine_labels"], dtype=np.int32)
        return images, labels

    required = (data_dir / "train", data_dir / "test", data_dir / "meta")
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError("Missing local CIFAR-100 files: " + ", ".join(missing))
    train_images, train_labels = load_batch(data_dir / "train")
    test_images, test_labels = load_batch(data_dir / "test")
    return train_images, train_labels, test_images, test_labels


## Augmentation thủ công

Cutout che ô 8×8 bằng màu trung bình train; Random Erasing thay vùng 2–15% bằng nhiễu pixel. MixUp và CutMix trộn cả ảnh và nhãn.


In [2]:
def _random_cutout(image: tf.Tensor, mean: tf.Tensor) -> tf.Tensor:
    """Mask an 8x8 square; the centre may lie near the image edge."""
    center_y = tf.random.uniform([], 0, IMAGE_SIZE, dtype=tf.int32)
    center_x = tf.random.uniform([], 0, IMAGE_SIZE, dtype=tf.int32)
    yy = tf.range(IMAGE_SIZE)[:, None]
    xx = tf.range(IMAGE_SIZE)[None, :]
    mask = tf.logical_and(
        tf.logical_and(yy >= center_y - 4, yy < center_y + 4),
        tf.logical_and(xx >= center_x - 4, xx < center_x + 4),
    )
    return tf.where(mask[:, :, None], mean[None, None, :], image)

def _random_erasing(image: tf.Tensor) -> tf.Tensor:
    """Erase a 2–15% area rectangle with pixel noise."""
    area = tf.random.uniform([], 0.02, 0.15) * (IMAGE_SIZE * IMAGE_SIZE)
    aspect = tf.random.uniform([], 0.5, 2.0)
    height = tf.clip_by_value(tf.cast(tf.round(tf.sqrt(area / aspect)), tf.int32), 1, IMAGE_SIZE)
    width = tf.clip_by_value(tf.cast(tf.round(tf.sqrt(area * aspect)), tf.int32), 1, IMAGE_SIZE)
    top = tf.random.uniform([], 0, IMAGE_SIZE - height + 1, dtype=tf.int32)
    left = tf.random.uniform([], 0, IMAGE_SIZE - width + 1, dtype=tf.int32)
    yy = tf.range(IMAGE_SIZE)[:, None]
    xx = tf.range(IMAGE_SIZE)[None, :]
    mask = tf.logical_and(
        tf.logical_and(yy >= top, yy < top + height),
        tf.logical_and(xx >= left, xx < left + width),
    )
    noise = tf.random.uniform(tf.shape(image), 0.0, 1.0)
    return tf.where(mask[:, :, None], noise, image)

def _sample_beta(alpha: float) -> tf.Tensor:
    a = tf.random.gamma([], alpha)
    b = tf.random.gamma([], alpha)
    return a / (a + b)

def _mixup(images: tf.Tensor, labels: tf.Tensor, alpha: float):
    indices = tf.random.shuffle(tf.range(tf.shape(images)[0]))
    lam = _sample_beta(alpha)
    return (
        lam * images + (1.0 - lam) * tf.gather(images, indices),
        lam * labels + (1.0 - lam) * tf.gather(labels, indices),
    )

def _cutmix(images: tf.Tensor, labels: tf.Tensor, alpha: float):
    indices = tf.random.shuffle(tf.range(tf.shape(images)[0]))
    lam = _sample_beta(alpha)
    patch = tf.cast(tf.round(tf.sqrt(1.0 - lam) * IMAGE_SIZE), tf.int32)
    center_y = tf.random.uniform([], 0, IMAGE_SIZE, dtype=tf.int32)
    center_x = tf.random.uniform([], 0, IMAGE_SIZE, dtype=tf.int32)
    y1 = tf.maximum(0, center_y - patch // 2)
    y2 = tf.minimum(IMAGE_SIZE, center_y + (patch + 1) // 2)
    x1 = tf.maximum(0, center_x - patch // 2)
    x2 = tf.minimum(IMAGE_SIZE, center_x + (patch + 1) // 2)
    yy = tf.range(IMAGE_SIZE)[:, None]
    xx = tf.range(IMAGE_SIZE)[None, :]
    mask = tf.logical_and(
        tf.logical_and(yy >= y1, yy < y2),
        tf.logical_and(xx >= x1, xx < x2),
    )[None, :, :, None]
    mixed_images = tf.where(mask, tf.gather(images, indices), images)
    kept = 1.0 - tf.cast((y2 - y1) * (x2 - x1), tf.float32) / (IMAGE_SIZE * IMAGE_SIZE)
    mixed_labels = kept * labels + (1.0 - kept) * tf.gather(labels, indices)
    return mixed_images, mixed_labels


## Pipeline train và validation

Mean/std tính riêng trên 45.000 ảnh train. Train dùng nhãn one-hot vì MixUp và CutMix tạo nhãn mềm.


In [3]:
def make_datasets(config: TrainConfig):
    train_images, train_labels, _, _ = load_cifar100()
    generator = np.random.default_rng(config.seed)
    permutation = generator.permutation(len(train_images))
    validation_indices = permutation[: config.validation_size]
    training_indices = permutation[config.validation_size :]
    train_x, train_y = train_images[training_indices], train_labels[training_indices]
    val_x, val_y = train_images[validation_indices], train_labels[validation_indices]

    mean = tf.constant(train_x.mean(axis=(0, 1, 2), dtype=np.float64) / 255.0, tf.float32)
    std = tf.constant(train_x.std(axis=(0, 1, 2), dtype=np.float64) / 255.0, tf.float32)
    autotune = tf.data.AUTOTUNE

    def augment(image, label):
        image = tf.cast(image, tf.float32) / 255.0
        image = tf.image.random_flip_left_right(image)
        image = tf.pad(image, [[4, 4], [4, 4], [0, 0]], mode="REFLECT")
        image = tf.image.random_crop(image, [IMAGE_SIZE, IMAGE_SIZE, 3])

        # Light colour jitter; one coin flip avoids distorting every training image.
        def jitter():
            x = tf.image.random_brightness(image, max_delta=0.05)
            x = tf.image.random_contrast(x, lower=0.9, upper=1.1)
            x = tf.image.random_saturation(x, lower=0.9, upper=1.1)
            x = tf.image.random_hue(x, max_delta=0.02)
            return tf.clip_by_value(x, 0.0, 1.0)

        image = tf.cond(tf.random.uniform([]) < 0.8, jitter, lambda: image)
        choice = tf.random.uniform([])
        image = tf.cond(
            choice < config.cutout_probability,
            lambda: _random_cutout(image, mean),
            lambda: tf.cond(
                choice < config.cutout_probability + config.erasing_probability,
                lambda: _random_erasing(image),
                lambda: image,
            ),
        )
        image = (image - mean) / std
        return image, tf.one_hot(label, NUM_CLASSES)

    def clean(image, label):
        image = (tf.cast(image, tf.float32) / 255.0 - mean) / std
        return image, tf.one_hot(label, NUM_CLASSES)

    def mix_batch(images, labels):
        choice = tf.random.uniform([])
        return tf.cond(
            choice < config.mix_probability,
            lambda: _mixup(images, labels, config.mix_alpha),
            lambda: tf.cond(
                choice < config.mix_probability + config.cutmix_probability,
                lambda: _cutmix(images, labels, config.cutmix_alpha),
                lambda: (images, labels),
            ),
        )

    train_ds = (
        tf.data.Dataset.from_tensor_slices((train_x, train_y))
        .shuffle(len(train_x), seed=config.seed + 1, reshuffle_each_iteration=True)
        .map(augment, num_parallel_calls=autotune)
        .batch(config.batch_size)
        .map(mix_batch, num_parallel_calls=autotune)
        .prefetch(autotune)
    )
    val_ds = (
        tf.data.Dataset.from_tensor_slices((val_x, val_y))
        .map(clean, num_parallel_calls=autotune)
        .batch(config.batch_size)
        .prefetch(autotune)
    )
    return train_ds, val_ds, mean, std


## Wide ResNet-16-4 từ đầu

Hai residual block cho mỗi stage, độ rộng 64–128–256, kết thúc bằng global average pooling.


In [5]:
def build_wrn_16_4() -> tf.keras.Model:
    """CIFAR-style Wide ResNet with two residual blocks per stage."""
    layers = tf.keras.layers

    def block(inputs, channels: int, stride: int):
        x = layers.BatchNormalization()(inputs)
        x = layers.Activation("relu")(x)
        shortcut = inputs
        if stride != 1 or inputs.shape[-1] != channels:
            shortcut = layers.Conv2D(channels, 1, strides=stride, padding="same", use_bias=False)(inputs)
        x = layers.Conv2D(channels, 3, strides=stride, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
        return layers.Add()([shortcut, x])

    inputs = tf.keras.Input((IMAGE_SIZE, IMAGE_SIZE, 3))
    x = layers.Conv2D(16, 3, padding="same", use_bias=False)(inputs)
    for channels, first_stride in ((64, 1), (128, 2), (256, 2)):
        x = block(x, channels, first_stride)
        x = block(x, channels, 1)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name="wrn_16_4_cifar100")

## Chạy thử 20 epoch

Checkpoint lưu theo `val_accuracy` tốt nhất. Train accuracy với nhãn mềm chỉ mang tính tham khảo; dùng validation accuracy/loss để đánh giá. Tập test chưa được sử dụng trong lượt thử.


In [11]:
def run_experiment(
    config: TrainConfig = TrainConfig(),
    *,
    smoke_test: bool = False,
    use_plateau_callbacks: bool = False,
):
    tf.keras.utils.set_random_seed(config.seed)
    train_ds, val_ds, mean, std = make_datasets(config)
    model = build_wrn_16_4()
    sample_images, sample_labels = next(iter(train_ds))
    assert sample_images.shape[1:] == (32, 32, 3)
    assert sample_labels.shape[-1] == NUM_CLASSES
    tf.debugging.assert_near(tf.reduce_sum(sample_labels, axis=1), tf.ones(tf.shape(sample_labels)[0]))
    tf.debugging.assert_all_finite(sample_images, "Augmentation produced non-finite images")

    steps_per_epoch = math.ceil((50_000 - config.validation_size) / config.batch_size)
    if use_plateau_callbacks:
        learning_rate = 1e-3
    else:
        learning_rate = tf.keras.optimizers.schedules.CosineDecay(
            initial_learning_rate=1e-3,
            decay_steps=steps_per_epoch * config.epochs,
            alpha=0.01,
        )
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=learning_rate, weight_decay=1e-4),
        loss=tf.keras.losses.CategoricalCrossentropy(),
        metrics=[tf.keras.metrics.CategoricalAccuracy(name="accuracy")],
    )
    print(f"Model parameters: {model.count_params():,}", flush=True)
    print(f"Train batches: {steps_per_epoch}; validation batches: {len(val_ds)}", flush=True)
    print(f"Training RGB mean: {mean.numpy()}; std: {std.numpy()}", flush=True)

    if smoke_test:
        result = model.train_on_batch(sample_images, sample_labels, return_dict=True)
        print(f"One-batch smoke test: {result}", flush=True)
        return result

    checkpoint_dir = PROJECT_DIR / "checkpoints"
    checkpoint_dir.mkdir(exist_ok=True)
    checkpoint_path = checkpoint_dir / f"cifar100_wrn16_4_aug{config.epochs}_best.keras"
    history_path = checkpoint_dir / f"cifar100_wrn16_4_aug{config.epochs}_history.csv"
    if use_plateau_callbacks:
        callbacks = [
            tf.keras.callbacks.ModelCheckpoint(
                filepath=str(checkpoint_path),
                monitor="val_loss",
                mode="min",
                save_best_only=True,
                verbose=1,
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                mode="min",
                factor=0.5,
                patience=3,
                min_delta=0.0,
                min_lr=1e-5,
                verbose=1,
            ),
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                mode="min",
                patience=20,
                restore_best_weights=True,
                verbose=1,
            ),
            tf.keras.callbacks.CSVLogger(str(history_path), append=False),
        ]
    else:
        callbacks = [
            tf.keras.callbacks.ModelCheckpoint(
                str(checkpoint_path), monitor="val_accuracy", mode="max", save_best_only=True
            ),
            tf.keras.callbacks.CSVLogger(str(history_path)),
        ]
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=config.epochs,
        callbacks=callbacks,
        verbose=2,
    )
    best_model = tf.keras.models.load_model(checkpoint_path)
    val_metrics = best_model.evaluate(val_ds, return_dict=True, verbose=0)
    print(f"Best checkpoint: {checkpoint_path}", flush=True)
    print(f"History: {history_path}", flush=True)
    print(f"Best validation metrics: {val_metrics}", flush=True)
    return history, val_metrics


In [7]:
config = TrainConfig(epochs=20)
print("TensorFlow:", tf.__version__)
print("Devices:", tf.config.list_physical_devices())
print("Data:", PROJECT_DIR / "cifar-100")
model = build_wrn_16_4()
print("Parameters:", f"{model.count_params():,}")
model.summary()


TensorFlow: 2.15.0
Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
Data: D:\22DT3\AI-Projects\fruit-detection\cifar-100


Parameters: 2,775,636
Model: "wrn_16_4_cifar100"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 32, 32, 3)]          0         []                            
                                                                                                  
 conv2d (Conv2D)             (None, 32, 32, 16)           432       ['input_1[0][0]']             
                                                                                                  
 batch_normalization (Batch  (None, 32, 32, 16)           64        ['conv2d[0][0]']              
 Normalization)                                                                                   
         

In [12]:
history, val_metrics = run_experiment(config)
print("Validation của checkpoint tốt nhất:", val_metrics)


Model parameters: 2,775,636
Train batches: 704; validation batches: 79
Training RGB mean: [0.5070944  0.48648465 0.4407329 ]; std: [0.26716948 0.25635508 0.27603567]
Epoch 1/20


KeyboardInterrupt: 

## Train tối đa 200 epoch

Chạy sau lượt thử 20 epoch nếu muốn huấn luyện đầy đủ. Cấu hình callback theo `build_cnn_cifar_100`: lưu checkpoint tốt nhất theo `val_loss`, giảm learning rate khi validation chững lại, dừng sớm sau 20 epoch không cải thiện và ghi CSV history. WRN vẫn dùng AdamW với weight decay `1e-4`; checkpoint 200 epoch được lưu riêng.


In [13]:
config_200 = TrainConfig(epochs=200)
history_200, val_metrics_200 = run_experiment(
    config_200,
    use_plateau_callbacks=True,
)
print("Validation của checkpoint 200 epoch tốt nhất:", val_metrics_200)


Model parameters: 2,775,636
Train batches: 704; validation batches: 79
Training RGB mean: [0.5070944  0.48648465 0.4407329 ]; std: [0.26716948 0.25635508 0.27603567]
Epoch 1/200

Epoch 1: val_loss improved from inf to 4.28385, saving model to D:\22DT3\AI-Projects\fruit-detection\checkpoints\cifar100_wrn16_4_aug200_best.keras
704/704 - 439s - loss: 4.0308 - accuracy: 0.0939 - val_loss: 4.2838 - val_accuracy: 0.1074 - lr: 0.0010 - 439s/epoch - 624ms/step
Epoch 2/200

Epoch 2: val_loss improved from 4.28385 to 3.28866, saving model to D:\22DT3\AI-Projects\fruit-detection\checkpoints\cifar100_wrn16_4_aug200_best.keras
704/704 - 436s - loss: 3.5620 - accuracy: 0.1788 - val_loss: 3.2887 - val_accuracy: 0.2100 - lr: 0.0010 - 436s/epoch - 620ms/step
Epoch 3/200

Epoch 3: val_loss improved from 3.28866 to 2.85142, saving model to D:\22DT3\AI-Projects\fruit-detection\checkpoints\cifar100_wrn16_4_aug200_best.keras
704/704 - 436s - loss: 3.2169 - accuracy: 0.2522 - val_loss: 2.8514 - val_accuracy:

## Đánh giá checkpoint trên test set

Đánh giá checkpoint có `val_loss` tốt nhất của lượt train 200 epoch trên 10.000 ảnh test CIFAR-100. Dùng lại mean/std của đúng 45.000 ảnh train và chỉ chuẩn hóa ảnh, không augmentation. Test set chỉ được dùng ở bước đánh giá cuối.


In [14]:
checkpoint_200_path = PROJECT_DIR / "checkpoints" / "cifar100_wrn16_4_aug200_best.keras"
if not checkpoint_200_path.is_file():
    raise FileNotFoundError(f"Không tìm thấy checkpoint: {checkpoint_200_path}")

test_config = TrainConfig(epochs=200)
train_images, _, test_images, test_labels = load_cifar100()
split_rng = np.random.default_rng(test_config.seed)
train_indices = split_rng.permutation(len(train_images))[test_config.validation_size:]
train_images_for_stats = train_images[train_indices]
train_mean = train_images_for_stats.mean(axis=(0, 1, 2), dtype=np.float64) / 255.0
train_std = train_images_for_stats.std(axis=(0, 1, 2), dtype=np.float64) / 255.0
del train_images, train_images_for_stats

def normalize_test_image(image, label):
    image = (tf.cast(image, tf.float32) / 255.0 - train_mean) / train_std
    return image, tf.one_hot(label, NUM_CLASSES)

test_ds = (
    tf.data.Dataset.from_tensor_slices((test_images, test_labels))
    .map(normalize_test_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(test_config.batch_size)
    .prefetch(tf.data.AUTOTUNE)
)
best_model_200 = tf.keras.models.load_model(str(checkpoint_200_path))
test_metrics = best_model_200.evaluate(test_ds, return_dict=True, verbose=1)
print(f"Test images: {len(test_labels):,}")
print("CIFAR-100 test metrics:", test_metrics)


157/157 [==============================] - 32s 203ms/step - loss: 0.9530 - accuracy: 0.7405
Test images: 10,000
CIFAR-100 test metrics: {'loss': 0.9530237317085266, 'accuracy': 0.7404999732971191}
